# Decision Tree Taxi Destination Notebook

Cached-data Kaggle workflow for Porto taxi destination prediction: load prebuilt preprocessing artifacts generated by `code/build_preprocessed_datasets.py`, train a Decision Tree model, validate with Haversine distance, and export a submission.


## Cached Data Paths

These model notebooks do not rebuild raw preprocessing. They expect the cached preprocessed Kaggle Dataset generated by `code/build_preprocessed_datasets.py`, including `metadata.json`, `validation/`, and `final/` matrices. Attach that dataset on Kaggle or set `PREPROCESSED_DATA_DIR`; model artifacts and submissions are written under `/kaggle/working`.


In [ ]:
import os
from pathlib import Path

PREPROCESSED_DATA_DIR = Path(
    os.environ.get(
        "PREPROCESSED_DATA_DIR",
        "/kaggle/input/datasets/louishogge/preprocessed-taxi-dataset/data",
    )
)
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessed data directory:", PREPROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Submission directory:", SUBMISSION_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())


## Imports

Load dependencies used by cached data loading, modeling, tuning, and plots.


In [ ]:
import gc
import json
import os
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")


## Hyperparameters

Set the reproducible run configuration and active model parameters. The best-model summary below records the reviewed Kaggle/W&B run so future reruns can be checked against the current accepted configuration.


In [ ]:
SEED = 42
MODEL_NAME = "Decision Tree"
MODEL_SLUG = "decision_tree"

RUN_HYPERPARAMETER_TUNING = False
# Keep tuning below Kaggle's 12-hour consecutive run limit.
TUNING_MAX_TRIALS = 80
TUNING_MAX_SECONDS = 11 * 60 * 60
TUNING_STRATEGY = "optuna_tpe"
INSTALL_OPTUNA_IF_MISSING = True

WANDB_PROJECT = "machine-learning"
WANDB_ENTITY = "ml-l"
WANDB_SECRET_NAME = "wandb-api-key"

SUBMISSION_FILE = f"submission_{MODEL_SLUG}.csv"
BEST_PARAMS_PATH = ARTIFACT_DIR / f"best_params_{MODEL_SLUG}.json"
MODEL_BUNDLE_PATH = ARTIFACT_DIR / f"model_bundle_{MODEL_SLUG}.joblib"

BEST_MODEL_PARAMS = {
    "max_depth": 20,
    "min_samples_split": 150,
    "min_samples_leaf": 20,
    "max_features": None,
    "ccp_alpha": 0.0
}

BEST_MODEL_RUN_SUMMARY = {
    "reviewed_on": "2026-07-07",
    "source": "W&B export 2026-07-07T21:07:56 plus Kaggle output artifacts",
    "trial_index": 32,
    "equivalent_best_trial_indices": [
        16,
        17,
        19,
        20,
        22,
        23,
        24,
        29,
        32,
        33,
        42,
        43,
        52,
        53,
        54,
        62,
        63,
        64,
        65,
        66,
        72,
        73,
        74,
        77,
        79
    ],
    "tuner_type": "optuna_tpe",
    "completed_trials": 80,
    "configured_max_trials": 80,
    "configured_max_seconds": 39600,
    "tuning_elapsed_seconds": 4040.199293851853,
    "search_space_configurations": 720,
    "unique_configurations_tested": 51,
    "duplicate_configuration_evaluations": 29,
    "all_configurations_tested": False,
    "all_search_values_observed": True,
    "additional_testing_recommended": False,
    "additional_testing_recommendation_basis": "Post-hoc review of all 382 finished W&B decision-tree rows in wandb_export_2026-07-08T10_14_03.504+02_00.csv found 300 unique historical configurations, 261 current-search-space configurations, 25 exact best repeats, and no higher-value unexplored decision-tree region.",
    "post_hoc_decision_tree_runs_reviewed": 382,
    "post_hoc_unique_decision_tree_configurations_reviewed": 300,
    "post_hoc_current_search_space_configurations_reviewed": 261,
    "validation_haversine_mean_km": 2.32772526509648,
    "validation_haversine_median_km": 1.362865072381726,
    "validation_haversine_p90_km": 5.040440199774206,
    "validation_haversine_std_km": 3.5644074065806834,
    "train_haversine_mean_km": 2.157979650015103,
    "train_haversine_median_km": 1.2946748210889671,
    "train_haversine_p90_km": 4.679890140391426,
    "train_haversine_std_km": 3.138903799353086,
    "generalization_gap_km": 0.16974561508137676,
    "fit_seconds": 130.22226548194885,
    "local_submission_all_mean_km": 2.553248096607002,
    "local_submission_all_median_km": 1.3713433107785702,
    "local_submission_all_std_km": 3.6129991590295023,
    "local_submission_all_p90_km": 6.266843078722837,
    "local_submission_public_mean_km": 2.3745831721909867,
    "local_submission_private_mean_km": 2.7319130210230176,
    "local_submission_n_trips": 320
}

# Local closing search around the reviewed best tree: depth=20, leaf=20, split=150, max_features=None.
TUNING_SEARCH_SPACE = {
    "max_depth": [18, 19, 20, 21, 22],
    "min_samples_split": [100, 125, 150, 175, 200],
    "min_samples_leaf": [10, 15, 20, 25, 30],
    "max_features": [None],
    "ccp_alpha": [0.0],
}

MODEL_PARAMS = BEST_MODEL_PARAMS.copy()


## Reproducibility

Seed random generators used by model training and tuning.


In [ ]:
def seed_everything(seed=SEED):
    """Seed Python and NumPy random generators."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
print("Seed:", SEED)


## Utility Constants

Define the cached target-column order used by the preprocessing helpers.

In [ ]:
TARGET_COLUMNS = ["END_LONG", "END_LAT"]

## Cached Preprocessed Dataset Helpers

Load and validate the shared preprocessed validation and final datasets.


In [ ]:
PREPROCESSED_SCHEMA_VERSION = 1
EXPECTED_CATEGORICAL_ENCODING_POLICY = {
    "CALL_TYPE": "one_hot",
    "ORIGIN_CALL": "missing_indicator_plus_log_count",
    "ORIGIN_STAND": "one_hot_missing_as_0",
    "TAXI_ID": "one_hot",
}


def require_existing_path(path, description):
    """Return an existing path or fail with an actionable Kaggle Dataset message."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}. "
            "Attach the preprocessed-taxi-dataset Kaggle Dataset or set PREPROCESSED_DATA_DIR."
        )
    return path


def load_preprocessed_metadata(base_dir=PREPROCESSED_DATA_DIR):
    """Load and validate cached preprocessing metadata."""
    metadata_path = require_existing_path(base_dir / "metadata.json", "preprocessed metadata")
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    schema_version = metadata.get("schema_version")
    if schema_version != PREPROCESSED_SCHEMA_VERSION:
        raise ValueError(
            f"Unsupported preprocessed dataset schema_version={schema_version!r}; "
            f"expected {PREPROCESSED_SCHEMA_VERSION}."
        )

    categorical_encoding_policy = metadata.get("constants", {}).get("categorical_encoding_policy")
    if categorical_encoding_policy != EXPECTED_CATEGORICAL_ENCODING_POLICY:
        raise ValueError(
            "Attached preprocessed dataset was built with an unexpected categorical encoding policy. "
            f"Expected {EXPECTED_CATEGORICAL_ENCODING_POLICY!r}, got {categorical_encoding_policy!r}. "
            "Regenerate the cached dataset with the current build_preprocessed_datasets.py script."
        )
    return metadata


def load_cached_frame(relative_path, base_dir=PREPROCESSED_DATA_DIR):
    """Load one cached Parquet frame from the preprocessed dataset."""
    frame_path = require_existing_path(base_dir / relative_path, f"cached frame {relative_path}")
    return pd.read_parquet(frame_path)


def assert_matching_features(left, right, left_name, right_name):
    """Validate that two feature matrices use the same columns in the same order."""
    if not left.columns.equals(right.columns):
        missing_from_right = list(left.columns.difference(right.columns))
        missing_from_left = list(right.columns.difference(left.columns))
        raise ValueError(
            f"Feature columns differ between {left_name} and {right_name}. "
            f"Missing from {right_name}: {missing_from_right}; missing from {left_name}: {missing_from_left}."
        )


def assert_index_aligned(features, targets, split_name):
    """Validate feature/target row alignment."""
    if not features.index.equals(targets.index):
        raise ValueError(f"Cached {split_name} features and targets have different indexes.")


def assert_target_columns(targets, split_name):
    """Validate target columns and order."""
    if list(targets.columns) != TARGET_COLUMNS:
        raise ValueError(
            f"Cached {split_name} target columns are {list(targets.columns)!r}; "
            f"expected {TARGET_COLUMNS!r}."
        )


def load_cached_validation_data():
    """Load cached train/validation matrices for fair model comparison."""
    metadata = load_preprocessed_metadata()
    validation_dir = require_existing_path(PREPROCESSED_DATA_DIR / "validation", "validation data directory")

    X_train = load_cached_frame("validation/X_train.parquet")
    y_train = load_cached_frame("validation/y_train.parquet")
    X_val = load_cached_frame("validation/X_val.parquet")
    y_val = load_cached_frame("validation/y_val.parquet")

    assert_index_aligned(X_train, y_train, "validation train")
    assert_index_aligned(X_val, y_val, "validation holdout")
    assert_matching_features(X_train, X_val, "X_train", "X_val")
    assert_target_columns(y_train, "validation train")
    assert_target_columns(y_val, "validation holdout")

    print("Loaded cached validation data from:", validation_dir)
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    print("Feature column count:", X_train.shape[1])
    return metadata, X_train, y_train, X_val, y_val


def load_cached_final_data():
    """Load cached full-train and competition-test matrices for submission."""
    final_dir = require_existing_path(PREPROCESSED_DATA_DIR / "final", "final data directory")
    scaler_path = require_existing_path(final_dir / "gps_scaler.joblib", "final GPS scaler")
    categorical_encoder_path = require_existing_path(
        final_dir / "categorical_encoder.joblib",
        "final categorical encoder",
    )

    X_final_train = load_cached_frame("final/X_train.parquet")
    y_final_train = load_cached_frame("final/y_train.parquet")
    X_final_test = load_cached_frame("final/X_test.parquet")
    final_gps_scaler = joblib.load(scaler_path)
    final_categorical_encoder = joblib.load(categorical_encoder_path)

    assert_index_aligned(X_final_train, y_final_train, "final train")
    assert_matching_features(X_final_train, X_final_test, "X_final_train", "X_final_test")
    assert_target_columns(y_final_train, "final train")

    print("Loaded cached final data from:", final_dir)
    print("X_final_train shape:", X_final_train.shape)
    print("y_final_train shape:", y_final_train.shape)
    print("X_final_test shape:", X_final_test.shape)
    return X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder


## Evaluation And Submission Helpers

Evaluate validation predictions and write the final Kaggle submission file.


In [ ]:
def haversine(pred, gt):
    """Return row-wise Haversine distances in kilometers for [LAT, LONG] arrays."""
    pred_lat = np.radians(pred[:, 0])
    pred_long = np.radians(pred[:, 1])
    gt_lat = np.radians(gt[:, 0])
    gt_long = np.radians(gt[:, 1])

    dlat = gt_lat - pred_lat
    dlon = gt_long - pred_long

    a = np.sin(dlat/2)**2 + np.cos(pred_lat) * \
        np.cos(gt_lat) * np.sin(dlon/2)**2

    d = 2 * 6371 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return d

def long_lat_to_lat_long(values):
    """Convert [LONG, LAT] arrays to [LAT, LONG] arrays."""
    values = np.asarray(values, dtype=float)
    return np.column_stack([values[:, 1], values[:, 0]])

def evaluate_long_lat_predictions(y_true_long_lat, y_pred_long_lat, prefix="val"):
    """Evaluate [LONG, LAT] predictions with Haversine distances."""
    distances = haversine(
        pred=long_lat_to_lat_long(y_pred_long_lat),
        gt=long_lat_to_lat_long(y_true_long_lat),
    )
    return {
        f"{prefix}/haversine_mean_km": float(distances.mean()),
        f"{prefix}/haversine_median_km": float(np.median(distances)),
        f"{prefix}/haversine_std_km": float(distances.std()),
        f"{prefix}/haversine_p90_km": float(np.percentile(distances, 90)),
    }, distances

def write_submission(trip_ids, destinations, output_path):
    """Write a Kaggle submission CSV from [LAT, LONG] destinations."""
    n_samples = len(trip_ids)
    assert destinations.shape == (n_samples, 2)

    submission = pd.DataFrame(
        data={
            'LATITUDE': destinations[:, 0],
            'LONGITUDE': destinations[:, 1],
        },
        columns=["LATITUDE", "LONGITUDE"],
        index=trip_ids,
    )

    submission.to_csv(output_path, index_label="TRIP_ID")


def make_json_safe(value):
    """Convert common notebook values to JSON-safe objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value

def save_json_artifact(payload, output_path):
    """Write an indented JSON artifact to Kaggle output storage."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(make_json_safe(payload), file, indent=2, sort_keys=True)

def save_best_params_artifact(tuning_results, validation_metrics):
    """Persist active parameters and their validation context."""
    payload = {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "params_source": "tuning_results" if not tuning_results.empty else "BEST_MODEL_PARAMS",
        "saved_best_model_params": BEST_MODEL_PARAMS,
        "best_model_run_summary": BEST_MODEL_RUN_SUMMARY,
        "active_model_params": MODEL_PARAMS,
        "validation_metrics": validation_metrics,
        "preprocessed_dataset": {
            "path": str(PREPROCESSED_DATA_DIR),
            "schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "config": PREPROCESSED_METADATA.get("config", {}),
            "validation": PREPROCESSED_METADATA.get("validation", {}),
            "final": PREPROCESSED_METADATA.get("final", {}),
        },
        "tuning": {
            "run_hyperparameter_tuning": RUN_HYPERPARAMETER_TUNING,
            "strategy": TUNING_STRATEGY,
            "max_trials": TUNING_MAX_TRIALS,
            "max_seconds": TUNING_MAX_SECONDS,
            "search_space": TUNING_SEARCH_SPACE,
        },
    }
    if not tuning_results.empty:
        best_trial = tuning_results.sort_values("val/haversine_mean_km", ascending=True).iloc[0]
        payload["best_trial"] = best_trial.to_dict()

    save_json_artifact(payload, BEST_PARAMS_PATH)
    return payload


## Cached Validation Dataset Loading

Load shared train/validation matrices for fair model comparison.


In [ ]:
PREPROCESSED_METADATA, X_train, y_train, X_val, y_val = load_cached_validation_data()

validation_metadata = PREPROCESSED_METADATA.get("validation", {})
prefix_summary = pd.DataFrame(validation_metadata.get("prefix_summary", {}))
if not prefix_summary.empty:
    display(prefix_summary)


## Define Decision Tree Model

Build the estimator used by validation, tuning, and final submission.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

def build_model(params=None):
    """Build the Decision Tree estimator with active parameters."""
    model_params = (params or MODEL_PARAMS).copy()
    model_params["random_state"] = SEED
    return DecisionTreeRegressor(**model_params)


## Hyperparameter Tuning

Optional tuning is disabled by default; saved best parameters are used unless explicitly enabled.


### Tuning Logging Helpers

Validate W&B once before optional tuning. Missing or invalid credentials disable logging, and per-trial SDK failures never interrupt model evaluation.

In [ ]:
WANDB_ENABLED = False
_WANDB_MODULE = None
_WANDB_MODE = "disabled"

def get_wandb_key():
    """Return the W&B API key from Kaggle secrets or the environment."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    except Exception:
        return os.environ.get("WANDB_API_KEY")

def _wandb_warning(stage, exception):
    """Report a sanitized W&B failure without exposing credentials."""
    print(
        f"W&B {stage} failed ({type(exception).__name__}). "
        "Model tuning will continue."
    )

def initialize_wandb():
    """Validate W&B credentials once before optional tuning starts."""
    wandb_key = get_wandb_key()
    if not wandb_key:
        print("No W&B key found. Tuning will run without W&B logging.")
        return None, "disabled", False

    try:
        import wandb
        login_succeeded = wandb.login(key=wandb_key)
        if login_succeeded is False:
            raise RuntimeError("W&B login was rejected.")
    except Exception as exception:
        _wandb_warning("setup", exception)
        return None, "disabled", False

    return wandb, "online", True

def sanitize_for_wandb(value):
    """Convert values to W&B-safe scalar or container types."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, dict):
        return {key: sanitize_for_wandb(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_for_wandb(item) for item in value]
    return value

def log_tuning_trial_to_wandb(result):
    """Log one tuning trial without allowing W&B to interrupt tuning."""
    if not WANDB_ENABLED:
        return

    run = None
    try:
        trial_index = int(result["trial_index"])
        params = sanitize_for_wandb(result["params"])
        tuner_type = result["tuner_type"]
        preprocessed_config = PREPROCESSED_METADATA.get("config", {})
        config = {
            "model_slug": MODEL_SLUG,
            "trial_index": trial_index,
            "trial_params": params,
            "seed": SEED,
            "tuner_type": tuner_type,
            "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "preprocessed_n_rows": preprocessed_config.get("n_rows"),
            "preprocessed_validation_size": preprocessed_config.get("validation_size"),
            "preprocessed_prefix_sample_factor": preprocessed_config.get("prefix_sample_factor"),
        }

        low_importance_metric_keys = {
            "train/haversine_median_km",
            "train/haversine_p90_km",
            "train/haversine_std_km",
        }
        metrics = {
            key: sanitize_for_wandb(value)
            for key, value in result.items()
            if key not in {"params", "params_repr", "tuner_type"}
            and key not in low_importance_metric_keys
        }

        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=f"{MODEL_SLUG}-{tuner_type}-trial-{trial_index:03d}",
            mode=_WANDB_MODE,
            config=config,
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no run.")
        _WANDB_MODULE.log(metrics)
    except Exception as exception:
        _wandb_warning("trial logging", exception)
    finally:
        if run is not None:
            try:
                run.finish()
            except Exception as exception:
                _wandb_warning("run finalization", exception)

if RUN_HYPERPARAMETER_TUNING:
    _WANDB_MODULE, _WANDB_MODE, WANDB_ENABLED = initialize_wandb()

### Tuning Trial Evaluation

Fit candidate configurations and rank them by validation Haversine distance.


In [ ]:
def evaluate_tuning_trial(trial_params, trial_index, tuner_type):
    """Fit one trial configuration and return validation metrics."""
    trial_model = build_model(params=trial_params)

    start_time = time.time()
    trial_model.fit(X_train, y_train)
    fit_seconds = time.time() - start_time

    train_predictions = trial_model.predict(X_train)
    val_predictions = trial_model.predict(X_val)

    train_metrics, _ = evaluate_long_lat_predictions(y_train.values, train_predictions, prefix="train")
    val_metrics, _ = evaluate_long_lat_predictions(y_val.values, val_predictions, prefix="val")

    return {
        "trial_index": trial_index,
        "params": trial_params.copy(),
        "params_repr": repr(trial_params),
        "tuner_type": tuner_type,
        **train_metrics,
        **val_metrics,
        "generalization/haversine_gap_km": float(
            val_metrics["val/haversine_mean_km"] - train_metrics["train/haversine_mean_km"]
        ),
        "fit_seconds": float(fit_seconds),
    }

def register_tuning_result(results, result):
    """Store, log, and print one tuning result."""
    previous_best = min(
        [row["val/haversine_mean_km"] for row in results],
        default=float("inf"),
    )
    result["trial/is_best_so_far"] = bool(result["val/haversine_mean_km"] <= previous_best)
    results.append(result)
    log_tuning_trial_to_wandb(result)
    print(
        f"  train={result['train/haversine_mean_km']:.4f} km, "
        f"val={result['val/haversine_mean_km']:.4f} km, "
        f"gap={result['generalization/haversine_gap_km']:.4f} km"
    )

def finalize_tuning_results(results, optuna_study=None):
    """Select the best tuning result and update MODEL_PARAMS."""
    global MODEL_PARAMS

    if len(results) == 0:
        raise RuntimeError("No hyperparameter trials were evaluated.")

    best_result = min(results, key=lambda row: row["val/haversine_mean_km"])
    MODEL_PARAMS = best_result["params"].copy()

    tuning_results = pd.DataFrame([
        {key: value for key, value in result.items() if key != "params"}
        for result in results
    ]).sort_values("val/haversine_mean_km", ascending=True)

    if optuna_study is not None:
        print(f"Optuna best value: {optuna_study.best_value:.6f} km")
    else:
        print(f"Best validation Haversine: {best_result['val/haversine_mean_km']:.6f} km")

    import pprint
    print("Paste-ready best config:")
    print("BEST_MODEL_PARAMS = " + pprint.pformat(MODEL_PARAMS, sort_dicts=False))
    display(tuning_results)
    return tuning_results


### Finite Search Tuning

Evaluate the deterministic shuffled subset used by random-grid tuning and the Optuna fallback.

In [ ]:
def get_finite_candidate_configs():
    """Return the deterministic shuffled finite candidate subset."""
    candidates = list(ParameterGrid(TUNING_SEARCH_SPACE))
    random.Random(SEED).shuffle(candidates)
    if TUNING_MAX_TRIALS is not None:
        candidates = candidates[:TUNING_MAX_TRIALS]
    return candidates

def run_finite_search_tuning(tuner_type):
    """Evaluate finite-grid candidates until trial or time limits are reached."""
    candidates = get_finite_candidate_configs()
    if len(candidates) == 0:
        raise ValueError("TUNING_SEARCH_SPACE produced no trials.")

    print(f"Starting {tuner_type} tuning for {MODEL_NAME} with {len(candidates)} trial(s).")
    tuning_start_time = time.time()
    results = []

    for trial_index, trial_params in enumerate(candidates, start=1):
        elapsed_seconds = time.time() - tuning_start_time
        if TUNING_MAX_SECONDS is not None and elapsed_seconds >= TUNING_MAX_SECONDS:
            print(f"Stopping tuning after {elapsed_seconds:.1f}s because TUNING_MAX_SECONDS was reached.")
            break

        print(f"Trial {trial_index}/{len(candidates)}: {trial_params}")
        result = evaluate_tuning_trial(trial_params, trial_index, tuner_type=tuner_type)
        result["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
        register_tuning_result(results, result)

    return finalize_tuning_results(results)

### Optuna Tuning

Use Optuna TPE for larger tree-based search spaces when enabled.


In [ ]:
def get_optuna_module():
    """Import or optionally install Optuna for TPE tuning."""
    try:
        import optuna
        return optuna
    except ImportError:
        if not INSTALL_OPTUNA_IF_MISSING:
            print("Optuna is not installed and INSTALL_OPTUNA_IF_MISSING is False.")
            return None

    try:
        import subprocess
        import sys
        print("Optuna is not installed. Attempting to install optuna...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
        import optuna
        return optuna
    except Exception as exc:
        print(f"Could not import or install Optuna ({exc}). Falling back to finite search.")
        return None

def get_optuna_startup_trials():
    """Return a conservative Optuna startup-trial count."""
    if TUNING_MAX_TRIALS is None:
        return 10
    return max(1, min(10, TUNING_MAX_TRIALS))

def suggest_params(trial):
    """Sample one parameter set from the shared tuning search space."""
    return {
        parameter_name: trial.suggest_categorical(parameter_name, values)
        for parameter_name, values in TUNING_SEARCH_SPACE.items()
    }

def run_optuna_tuning():
    """Run Optuna tuning and return the ranked trial results."""
    if TUNING_MAX_TRIALS is None and TUNING_MAX_SECONDS is None:
        raise ValueError("Set TUNING_MAX_TRIALS, TUNING_MAX_SECONDS, or both before running Optuna tuning.")

    optuna = get_optuna_module()
    if optuna is None:
        return run_finite_search_tuning(tuner_type="finite_fallback")

    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=get_optuna_startup_trials(),
    )
    study = optuna.create_study(direction="minimize", sampler=sampler)
    results = []
    tuning_start_time = time.time()

    def objective(trial):
        """Evaluate one Optuna trial and return validation distance."""
        trial_index = trial.number + 1
        trial_params = suggest_params(trial)
        print(f"Trial {trial_index}: {trial_params}")
        result = evaluate_tuning_trial(
            trial_params=trial_params,
            trial_index=trial_index,
            tuner_type="optuna_tpe",
        )
        result["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
        register_tuning_result(results, result)
        return result["val/haversine_mean_km"]

    print(
        f"Starting Optuna TPE tuning for {MODEL_NAME} "
        f"with max_trials={TUNING_MAX_TRIALS} and max_seconds={TUNING_MAX_SECONDS}."
    )
    study.optimize(
        objective,
        n_trials=TUNING_MAX_TRIALS,
        timeout=TUNING_MAX_SECONDS,
        show_progress_bar=True,
    )
    return finalize_tuning_results(results, optuna_study=study)

### Run Hyperparameter Tuning

Use saved best parameters unless tuning is explicitly enabled.


In [ ]:
def run_hyperparameter_tuning():
    """Run the configured tuning strategy or keep saved parameters."""
    global MODEL_PARAMS

    if not RUN_HYPERPARAMETER_TUNING:
        MODEL_PARAMS = BEST_MODEL_PARAMS.copy()
        print("Hyperparameter tuning disabled. Using BEST_MODEL_PARAMS:")
        print(MODEL_PARAMS)
        return pd.DataFrame()

    if TUNING_STRATEGY == "random_grid":
        return run_finite_search_tuning(tuner_type=TUNING_STRATEGY)
    if TUNING_STRATEGY == "optuna_tpe":
        return run_optuna_tuning()
    raise ValueError(f"Unsupported TUNING_STRATEGY: {TUNING_STRATEGY}")

tuning_results = run_hyperparameter_tuning()
print("Active MODEL_PARAMS:", MODEL_PARAMS)

## Train And Validate

Fit the active model and report train/validation Haversine metrics.


In [ ]:
model = build_model()
print(model)

start_time = time.time()
model.fit(X_train, y_train)
fit_seconds = time.time() - start_time

train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

train_metrics, _ = evaluate_long_lat_predictions(y_train.values, train_predictions, prefix="train")
val_metrics, val_distances = evaluate_long_lat_predictions(y_val.values, val_predictions, prefix="val")

train_val_metrics = {
    **train_metrics,
    **val_metrics,
    "generalization/haversine_gap_km": float(
        val_metrics["val/haversine_mean_km"] - train_metrics["train/haversine_mean_km"]
    ),
    "fit_seconds": float(fit_seconds),
}

print(pd.Series(train_val_metrics))

save_best_params_artifact(tuning_results, train_val_metrics)
print("Best params JSON written to:", BEST_PARAMS_PATH.resolve())

## Train vs Validation Plot

Compare train and validation Haversine errors with bounded diagnostic plots.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean_values = [
    train_metrics["train/haversine_mean_km"],
    val_metrics["val/haversine_mean_km"],
]
bars = axes[0].bar(["Train", "Validation"], mean_values, color=["#4C78A8", "#F58518"])
axes[0].set_ylabel("Mean Haversine distance (km)")
axes[0].set_title(f"{MODEL_NAME} train vs validation error")
axes[0].bar_label(bars, fmt="%.3f km")

axes[1].hist(val_distances, bins=40, color="#F58518")
axes[1].axvline(val_distances.mean(), color="red", linestyle="--", label=f"Mean: {val_distances.mean():.3f} km")
axes[1].set_xlabel("Haversine distance (km)")
axes[1].set_ylabel("Validation trips")
axes[1].set_title(f"{MODEL_NAME} validation error distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

for name in [
    "model",
    "train_predictions",
    "val_predictions",
    "val_distances",
    "X_train",
    "y_train",
    "X_val",
    "y_val",
]:
    globals().pop(name, None)
gc.collect()

## Final Fit And Kaggle Submission

Load cached full-train and test matrices, fit the final model, and write the submission file.


In [ ]:
X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder = load_cached_final_data()

final_model = build_model()
final_model.fit(X_final_train, y_final_train)

selected_best_run_summary = BEST_MODEL_RUN_SUMMARY.copy()
selected_best_trial_index = int(selected_best_run_summary["trial_index"])
equivalent_best_trial_indices = selected_best_run_summary.get(
    "equivalent_best_trial_indices", [selected_best_trial_index]
).copy()

model_bundle = {
    "model": final_model,
    "gps_scaler": final_gps_scaler,
    "categorical_encoder": final_categorical_encoder,
    "feature_columns": list(X_final_train.columns),
    "model_params": MODEL_PARAMS.copy(),
    "validation_metrics": train_val_metrics.copy(),
    "selected_best_run_summary": selected_best_run_summary,
    "selected_best_trial_index": selected_best_trial_index,
    "equivalent_best_trial_indices": equivalent_best_trial_indices,
    "final_refit": True,
    "final_refit_note": "Model is refit on final training data with the selected best parameters; it is not the serialized validation-trial estimator.",
    "prediction_format": "[LONG, LAT]",
    "preprocessed_data_dir": str(PREPROCESSED_DATA_DIR),
    "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
    "preprocessed_metadata": PREPROCESSED_METADATA,
}
joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
print("Model bundle written to:", MODEL_BUNDLE_PATH.resolve())

test_predictions = final_model.predict(X_final_test)

submission_path = SUBMISSION_DIR / SUBMISSION_FILE
test_destinations = long_lat_to_lat_long(test_predictions)
write_submission(X_final_test.index, test_destinations, submission_path)
submission = pd.read_csv(submission_path)

print("Submission written to:", submission_path.resolve())
display(submission.head())